# 📒 Index Expensify/App Files for LLM Context

In [102]:

import os
import re
import regex
from pathlib import Path
import json
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
SRC_DIR = Path("../repos/App/src").resolve()
print("SRC_DIR:", SRC_DIR)
    

SRC_DIR: /Users/rushatgabhane/code/blame-gpt/repos/App/src


In [155]:

def _should_ignore(path):
    IGNORE_DIRS = {
        "src/types",
        "src/styles",
        "src/stories",
        "src/setup",
        "src/languages",
        "src/utils",
        "src/libs/API/parameters",
    }
    IGNORE_FILES = {
        "src/libs/DateUtils.ts",
    }

    if any(path.startswith(ignored_dir) for ignored_dir in IGNORE_DIRS):
        return True
    if path in IGNORE_FILES:
        return True
    return False



def _extract_functions_and_jsdocs(content):
    functions = []
    pattern = re.compile(
        r"(\/\*\*[\s\S]*?\*\/)?\s*(?:function\s+(\w+)|const\s+(\w+)\s*=\s*\([^\)]*\)\s*=>|(\w+)\s*:\s*function\s*\([^\)]*\))",
        re.MULTILINE,
    )
    for match in pattern.finditer(content):
        jsdoc = match.group(1) or ""
        name = match.group(2) or match.group(3) or match.group(4)
        if name:
            functions.append((name, jsdoc.strip()))
    return functions

def _extract_jsx_return_block(content):
    blocks = []

    # Match: return (...) or return <Fragment>...</Fragment>
    paren_return_pattern = r'return\s*\(([\s\S]+?)\);'
    ternary_pattern = r'return\s+[^?]*\?\s*\(([\s\S]+?)\)\s*:\s*\(([\s\S]+?)\);'

    blocks += regex.findall(paren_return_pattern, content, flags=regex.DOTALL)

    ternary_matches = regex.findall(ternary_pattern, content, flags=regex.DOTALL)
    for left, right in ternary_matches:
        blocks.append(left)
        blocks.append(right)

    all_lines = []
    for block in blocks:
        cleaned = block.replace('\\n', '\n').replace("\\'", "'").replace('\\"', '"')
        lines = [line.strip() for line in cleaned.split('\n') if line.strip()]
        all_lines.extend(lines)

    return all_lines



def _remove_stuff_from_file(content):
    # Remove imports
    content = "\n".join(
        [line for line in content.splitlines() if not line.strip().startswith("import")]
    )

    # Remove eslint disables
    content = re.sub(r"//\s*eslint-disable.*", "", content)
    content = re.sub(r"^\s*$", "", content, flags=re.MULTILINE)

    content = regex.sub(r'Onyx\.connect\(\{[\s\S]*?\}\);', '', content)
    content = re.sub(r'type\s+\w+\s*=\s*{[^}]*}', '', content, flags=re.DOTALL)

    lines = content.splitlines()
    cleaned_lines = [line for line in lines if line.strip() and line.strip() != ";"]
    return "\n".join(cleaned_lines)


In [156]:

file_summaries = {}

for root, _, files in os.walk(SRC_DIR):
    for file in files:
        file_path = Path(root) / file
        rel_path = str(file_path)
        match = re.search(r"(src/.*)", rel_path)
        if match:
            rel_path = match.group(1)

        if rel_path.endswith("types.ts") or _should_ignore(rel_path):
            continue

        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()

        content = _remove_stuff_from_file(content)

        if rel_path.startswith("src/libs/"):
            functions = _extract_functions_and_jsdocs(content)

            formatted = []
            for name, jsdoc in functions:
                if jsdoc:
                    formatted.extend(jsdoc.strip().split('\n'))
                formatted.append(f"function {name}(...)")  # Use (...) as placeholder
            file_summaries[rel_path] = formatted

        elif file.endswith(".tsx"):
            jsx_return = _extract_jsx_return_block(content)
            file_summaries[rel_path] = jsx_return
        else:
            file_summaries[rel_path] = content.splitlines()[:150]
    

In [166]:
file_summaries['src/components/SelectionList/BaseSelectionList.tsx']

[') => removeKeyDownPressListener(setHasKeyBeenPressed',
 '// Note: The `optionsListSectionHeader` style provides an explicit height to section headers.',
 '// We do this so that we can reference the height in `getItemLayout` –',
 '// we need to know the heights of all list items up-front in order to synchronously compute the layout of any given list item.',
 '// So be aware that if you adjust the content of the section header (for example, change the font size), you may need to adjust this explicit height as well.',
 '<View style={[styles.optionsListSectionHeader, styles.justifyContentCenter, sectionTitleStyles]}>',
 '<Text style={[styles.ph5, styles.textLabelSupporting]}>{section.title}</Text>',
 '</View>',
 '<View onLayout={(event: LayoutChangeEvent) => onItemLayout(event, item?.keyForList)}>',
 '<BaseSelectionListItemRenderer',
 'ListItem={ListItem}',
 'item={{',
 'shouldAnimateInHighlight: isItemHighlighted,',
 '...item,',
 '}}',
 'shouldUseDefaultRightHandSideCheckmark={shouldUse

In [167]:

df_summary = pd.DataFrame([
    {"File": path, "Summary": str(summary)[:500]} for path, summary in file_summaries.items()
])

# file_summaries['src/libs/actions/Delegate.ts']

sorted_summaries = sorted(file_summaries.items(), key=lambda item: len(item[1]), reverse=True)

# Print each file path and number of lines, followed by its content
for path, summary in sorted_summaries:
    if len(summary) :
        # print(f"\n📄 {path} ({len(summary)} lines)")



_IncompleteInputError: incomplete input (4077856795.py, line 13)

In [168]:

# Optional: Save to JSON
with open("file_summaries.json", "w", encoding="utf-8") as f:
    json.dump(file_summaries, f, indent=2, ensure_ascii=False)
    